# 🏠 House Prices Prediction — chạy toàn bộ pipeline

Notebook này là điểm vào chính của dự án, tương đương `python main.py`.

**Luồng:** Clean → Feature → Preprocess → 5-fold OOF từng model → Ensemble → `submission.csv`

Trước khi chạy lần đầu:

```
pip install -r requirements.txt
pip install -e .
```

và đặt `train.csv`, `test.csv` từ Kaggle vào `data/raw/`.

In [1]:
# ===== Thiết lập =====
import sys, warnings
from pathlib import Path

# Notebook này nằm ở gốc dự án nên import được ngay.
# Dòng dưới là phương án dự phòng nếu bạn chưa chạy `pip install -e .`
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Tự nạp lại code khi file trong src/ thay đổi.
# Thiếu hai dòng này thì bạn sửa src/ mà notebook vẫn chạy bản cũ.
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 100)
warnings.filterwarnings('ignore', category=FutureWarning)

print('Thư mục dự án:', ROOT)

Thư mục dự án: C:\Users\Hung\OneDrive\ドキュメント\May_hoc\RepoNhom\Challege 1\2_HousePrices


## Tham số chạy

Sửa ba biến dưới đây thay cho các cờ dòng lệnh của `main.py`.

In [2]:
MEMBER = 'team'          # tên bạn, để log ghi riêng cho từng người
CONFIG = None            # None = config/global_config.yaml
                         # hoặc 'config/config_<ten_ban>.yaml'
MODELS = None            # None = lấy danh sách từ run.models trong config
                         # hoặc ['lasso', 'elasticnet']

from src.pipeline.Pipeline import HousePricePipeline, available_models
print('Model dùng được trên máy này:', available_models())

Model dùng được trên máy này: ['lasso', 'elasticnet', 'kernelridge', 'randomforest']


## Cách 1 — chạy một lệnh

Giống hệt `python main.py --member <ten>`. Xong là có `submission.csv`.

In [3]:
out = HousePricePipeline(CONFIG, MEMBER).run(MODELS)
print('\nFile nộp Kaggle:', out)

Bỏ qua model chưa khả dụng (thiếu thư viện?): ['lightgbm']


Dữ liệu sau xử lý: train (1458, 269), test (1459, 269)


lasso_{'rmse_oof': 0.11000236101985193, 'rmse_cv_mean': 0.10978424464255228, 'rmse_cv_std': 0.00685118391129233}


elasticnet_{'rmse_oof': 0.11099884257479493, 'rmse_cv_mean': 0.11077304077244157, 'rmse_cv_std': 0.0070171258124171335}


kernelridge_{'rmse_oof': 0.11051894362056526, 'rmse_cv_mean': 0.11024687774511055, 'rmse_cv_std': 0.007665798520944081}


randomforest_{'rmse_oof': 0.13323669404740635, 'rmse_cv_mean': 0.13288666020714382, 'rmse_cv_std': 0.009635049088592751}



       model  rmse_oof  rmse_cv_mean  rmse_cv_std
       lasso   0.11000       0.10978      0.00685
 kernelridge   0.11052       0.11025      0.00767
  elasticnet   0.11100       0.11077      0.00702
randomforest   0.13324       0.13289      0.00964



               method  rmse_oof                                                                            weights
             stacking   0.10865 {'lasso': 0.288, 'elasticnet': 0.263, 'kernelridge': 0.274, 'randomforest': 0.201}
             blending   0.10894 {'lasso': 0.295, 'elasticnet': 0.291, 'kernelridge': 0.287, 'randomforest': 0.127}
               voting   0.10947     {'lasso': 0.25, 'elasticnet': 0.25, 'kernelridge': 0.25, 'randomforest': 0.25}
       (đơn lẻ) lasso   0.11000                                                                                 {}
 (đơn lẻ) kernelridge   0.11052                                                                                 {}
  (đơn lẻ) elasticnet   0.11100                                                                                 {}
(đơn lẻ) randomforest   0.13324                                                                                 {}


ensemble_{'method': 'blending', 'rmse_oof': 0.10894313339881276, 'weights': {'lasso': 0.295, 'elasticnet': 0.291, 'kernelridge': 0.2869, 'randomforest': 0.1271}}


{'submission_rows': 1459, 'price_min': 45416.33115328673, 'price_max': 967224.9458271697}



File nộp Kaggle: C:\Users\Hung\OneDrive\ドキュメント\May_hoc\RepoNhom\Challege 1\2_HousePrices\SUBMIT-summarize_report\submission.csv


---

## Cách 2 — chạy từng bước

Phần dưới làm đúng những gì `run()` làm, nhưng tách ra để xem số liệu ở giữa.
Dùng khi cần gỡ lỗi, hoặc lấy bảng và biểu đồ cho báo cáo.

### Bước 1: Clean → Feature → Preprocess

In [4]:
pipe = HousePricePipeline(CONFIG, MEMBER)
X, y, X_test, test_ids = pipe.prepare()

print('X      :', X.shape)
print('X_test :', X_test.shape)
print('y (log):', round(y.min(), 3), '→', round(y.max(), 3))
assert list(X.columns) == list(X_test.columns), 'train và test lệch cột'
print('Train và test cùng bộ cột: OK')
X.head()

X      : (1458, 269)
X_test : (1459, 269)
y (log): 10.46 → 13.534
Train và test cùng bộ cột: OK


,LotFrontage,LotArea,LotShape,LandSlope,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,ExterQual,ExterCond,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,HeatingQC,CentralAir,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolQC,...,Heating_Floor,Heating_GasA,Heating_GasW,Heating_Grav,Heating_OthW,Heating_Wall,Electrical_FuseA,Electrical_FuseF,Electrical_FuseP,Electrical_Mix,Electrical_SBrkr,GarageType_2Types,GarageType_Attchd,GarageType_Basment,GarageType_BuiltIn,GarageType_CarPort,GarageType_Detchd,GarageType_None,MiscFeature_Gar2,MiscFeature_None,MiscFeature_Othr,MiscFeature_Shed,MiscFeature_TenC,MoSold_1,MoSold_10,MoSold_11,MoSold_12,MoSold_2,MoSold_3,MoSold_4,MoSold_5,MoSold_6,MoSold_7,MoSold_8,MoSold_9,SaleType_COD,SaleType_CWD,SaleType_Con,SaleType_ConLD,SaleType_ConLI,SaleType_ConLw,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial
0,-0.257516,-0.266140,0.0,0.0,0.5,0.0,0.663043,0.243243,1.035024,1.0,0.0,0.0,0.0,0.0,0.4,0.093330,0.0,0.0,-0.898157,-0.269192,0.0,0.0,-0.522512,1.024186,0.0,0.346313,1.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,0.0,-1.0,-0.50,0.595238,0.0,0.0,0.280164,0.0,0.0,0.0,0.000000,0.209879,0.000000,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.464671,0.030467,0.0,0.0,0.0,3.0,0.076087,-0.486486,0.000000,0.0,0.0,0.0,0.0,3.0,0.2,0.142888,0.0,0.0,-0.403505,0.540379,0.0,0.0,0.329826,0.000000,0.0,-0.323635,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,-0.047619,0.0,0.0,-0.079755,0.0,0.0,0.0,1.111220,-0.764857,0.000000,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.100761,0.399160,-1.0,0.0,0.5,0.0,0.619565,0.216216,0.997909,1.0,0.0,0.0,0.0,1.0,0.4,0.036576,0.0,0.0,-0.074216,-0.141575,0.0,0.0,-0.364219,1.026301,0.0,0.442219,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.25,0.547619,0.0,0.0,0.525562,0.0,0.0,0.0,0.000000,0.123453,0.000000,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,-0.535329,0.018328,-1.0,0.0,0.5,0.0,-1.250000,-0.648649,0.000000,0.0,0.0,-1.0,1.0,0.0,0.2,-0.086499,0.0,0.0,0.095602,-0.468594,-0.5,0.0,-0.268494,1.005718,0.0,0.355322,1.0,0.0,-1.0,0.0,0.0,0.0,1.0,0.5,0.0,0.0,0.50,0.476190,-1.0,1.0,0.664622,0.0,0.0,0.0,0.000000,0.081489,5.609472,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
4,0.634651,0.950309,-1.0,0.0,1.0,0.0,0.597826,0.162162,1.148177,1.0,0.0,0.0,0.0,2.0,0.4,0.081931,0.0,0.0,0.020085,0.307079,0.0,0.0,0.116169,1.055930,0.0,0.900045,1.0,0.0,0.0,1.0,1.0,0.0,1.0,1.5,0.0,0.0,0.25,0.523810,0.0,1.0,1.458078,0.0,0.0,0.0,1.025886,0.284397,0.000000,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


### Bước 2: cross-validation từng model

`oof_predict` chia 5 fold. Mỗi dòng train được dự đoán bởi model **chưa từng
thấy dòng đó**, nên con số này đáng tin hơn một lần chia train/test.
Với 1460 dòng, một lần chia cho sai số may rủi tới 0.01 RMSE.

In [5]:
from src.evaluation.Evaluation import Evaluator
from src.pipeline.Pipeline import MODELS as MODEL_REGISTRY

names = MODELS or pipe.cfg.get('run', {}).get('models') or available_models()
names = [n for n in names if n in MODEL_REGISTRY]

results = {}
for n in names:
    results[n] = Evaluator.oof_predict(
        lambda k=n: MODEL_REGISTRY[k](pipe.cfg), X, y, X_test,
        n_folds=pipe.cfg['split']['n_folds'], seed=pipe.cfg['project']['seed'])
    r = results[n]
    print(f"{n:14s} rmse_oof={r['rmse_oof']:.5f}   cv={r['rmse_mean']:.5f} ± {r['rmse_std']:.5f}")

lasso          rmse_oof=0.11000   cv=0.10978 ± 0.00685


elasticnet     rmse_oof=0.11100   cv=0.11077 ± 0.00702


kernelridge    rmse_oof=0.11052   cv=0.11025 ± 0.00767


randomforest   rmse_oof=0.13324   cv=0.13289 ± 0.00964


In [6]:
Evaluator.compare(results)

,model,rmse_oof,rmse_cv_mean,rmse_cv_std
0,lasso,0.11000,0.10978,0.00685
1,kernelridge,0.11052,0.11025,0.00767
2,elasticnet,0.11100,0.11077,0.00702
3,randomforest,0.13324,0.13289,0.00964


### Bước 3: so sánh các cách kết hợp model

Bảng này là nội dung chính cho phần ensemble của báo cáo.

Lưu ý khi đọc: model nào bị gán trọng số gần 0 nghĩa là nó không thêm thông tin
so với các model còn lại, thường vì sai theo cùng một kiểu.

In [7]:
from src.ensemble.Ensemble import compare_methods

oof = {k: v['oof'] for k, v in results.items()}
compare_methods(oof, y) if len(oof) > 1 else print('Cần ít nhất 2 model để trộn')

,method,rmse_oof,weights
0,stacking,0.10865,"{'lasso': 0.288, 'elasticnet': 0.263, 'kernelr..."
1,blending,0.10894,"{'lasso': 0.295, 'elasticnet': 0.291, 'kernelr..."
2,voting,0.10947,"{'lasso': 0.25, 'elasticnet': 0.25, 'kernelrid..."
3,(đơn lẻ) lasso,0.11000,{}
4,(đơn lẻ) kernelridge,0.11052,{}
5,(đơn lẻ) elasticnet,0.11100,{}
6,(đơn lẻ) randomforest,0.13324,{}


### Bước 4: biểu đồ chẩn đoán

Phần dư nên rải đều quanh 0. Nếu thấy hình phễu (càng về giá cao càng lệch)
thì model còn thiếu đặc trưng cho nhà đắt.

In [8]:
from src.utils.data_loader import get_path

best = min(results, key=lambda k: results[k]['rmse_oof'])
print('Model tốt nhất:', best, '| rmse_oof =', round(results[best]['rmse_oof'], 5))

figs = get_path(pipe.cfg, 'figures')
Evaluator.plot_pred_vs_actual(y, oof[best], fig_dir=figs, fname=f'pred_vs_actual_{best}.png')
Evaluator.plot_residuals(y, oof[best], fig_dir=figs, fname=f'residuals_{best}.png')

Model tốt nhất: lasso | rmse_oof = 0.11


### Bước 5: đặc trưng nào quan trọng

In [9]:
model = MODEL_REGISTRY[best](pipe.cfg).fit(X, y)
try:
    imp = model.feature_importance(X.columns)
    Evaluator.plot_feature_importance(imp, k=25, fig_dir=figs,
                                      fname=f'importance_{best}.png')
    display(imp.head(20).to_frame('importance'))
except AttributeError as e:
    print('Model này không báo được độ quan trọng:', e)

,importance
MSZoning_C (all),0.295423
Neighborhood_Crawfor,0.093703
TotRmsAbvGrd,0.079267
Neighborhood_StoneBr,0.075544
HouseAge,0.074076
TotalFinSF,0.067845
SaleType_New,0.058462
OverallQual,0.056508
Exterior1st_BrkFace,0.054521
LivAreaPerRoom,0.052451


---

## Sau khi chạy

- [ ] Nộp `SUBMIT-summarize_report/submission.csv` lên Kaggle
- [ ] Ghi một dòng vào `reports/results_summary.md`: ngày, người, model, RMSE OOF, điểm Kaggle
- [ ] Đối chiếu: điểm Kaggle lệch dưới 0.01 so với RMSE OOF là bình thường.
      Lệch nhiều hơn thì nghi có rò rỉ dữ liệu, phải tìm trước khi làm tiếp.

**Độ lệch chuẩn giữa các fold khoảng 0.007.** Mọi cải thiện nhỏ hơn con số đó
chưa thể kết luận là thật — muốn chắc thì chạy lại với `seed` khác trong config.